In [1]:
import os
import ast
import pandas as pd
from scipy.stats import randint, uniform
from sklearn.model_selection import RandomizedSearchCV, GroupKFold, GroupShuffleSplit
from sklearn.preprocessing import LabelEncoder
import xgboost as xgb
from datetime import datetime as dt

# 1. Load Raw/Full Feature Dataset
df = pd.read_csv('../dataset/full_v2.csv')  # Adjust path to your full dataset

# Separate features, target, and subject groups
X = df.drop(columns=['Activity', 'subject'])
y = df['Activity']
groups = df['subject']

# Integer-encode target labels for XGBoost
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# 2. Keep 20% of subjects as pure hold-out test set
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=69)
train_idx, test_idx = next(gss.split(X, y_encoded, groups=groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y_encoded[train_idx], y_encoded[test_idx]
groups_train, groups_test = groups.iloc[train_idx], groups.iloc[test_idx]

# 3. Expanded Continuous Parameter Distributions for Raw Features
param_distributions = {
    'n_estimators': randint(150, 500),         # More trees for 500+ features
    'max_depth': randint(3, 9),                 # Broader depth range
    'learning_rate': uniform(0.01, 0.15),       # Continuous learning pace
    'subsample': uniform(0.6, 0.4),              # Row sampling (60% to 100%)
    'colsample_bytree': uniform(0.3, 0.6),       # Feature sampling per tree (30% to 90%)
    'colsample_bylevel': uniform(0.5, 0.5),      # Feature sampling per depth level
    'min_child_weight': randint(1, 10),         # Prevents overfitting on noise
    'gamma': uniform(0, 1.0),                   # Minimum loss reduction
    'reg_alpha': uniform(0, 5.0),               # L1 Regularization for high-dim feature selection
    'reg_lambda': uniform(1.0, 10.0)            # L2 Regularization
}

# 4. Base Estimator Setup
xgb_model = xgb.XGBClassifier(
    objective='multi:softprob',
    eval_metric='mlogloss',
    tree_method='hist',                         # Essential for fast histogram binning
    random_state=69,
    n_jobs=1                                    # Single-threaded inside each job process
)

# 5. Group Cross-Validation
gkf = GroupKFold(n_splits=5)

search = RandomizedSearchCV(
    estimator=xgb_model,
    param_distributions=param_distributions,
    n_iter=80,                                  # 80 iterations x 5 folds = 400 fits (~85 mins)
    scoring='f1_macro',
    cv=gkf,
    verbose=2,                                  # Detailed log updates
    random_state=69,
    n_jobs=8                                    # Uses 8 M1 Performance cores cleanly
)

print(f"Starting 2-hour tuning search across {X_train.shape[1]} features...")
search.fit(X_train, y_train, groups=groups_train)

# 6. Auto-Save Results when finished
now_str = dt.now().strftime("%Y-%m-%d_%H-%M-%S")
os.makedirs('../dataset/results', exist_ok=True)

# Save best parameters
best_params_df = pd.DataFrame([{
    'best_score_f1_macro': search.best_score_,
    'best_params': search.best_params_
}])
best_params_df.to_csv(f'../dataset/results/full_dataset_best_params_{now_str}.csv', index=False)

# Evaluate on unseen hold-out subjects immediately
best_model = search.best_estimator_
y_pred = best_model.predict(X_test)

results_df = pd.DataFrame({
    'Subject': groups_test.values,
    'Actual_Activity': le.inverse_transform(y_test),
    'Predicted_Activity': le.inverse_transform(y_pred)
})
results_df.to_csv(f'../dataset/results/pred_full_dataset_raw_features_{now_str}_xgboost.csv', index=False)

print("\nSUCCESS! Models trained and evaluations saved.")
print(f"Best Training Group Macro-F1: {search.best_score_:.4f}")

Starting 2-hour tuning search across 540 features...
Fitting 5 folds for each of 80 candidates, totalling 400 fits
[CV] END colsample_bylevel=0.6481245808362168, colsample_bytree=0.785440629403996, gamma=0.35025252522341144, learning_rate=0.12841138846463387, max_depth=4, min_child_weight=9, n_estimators=375, reg_alpha=0.5248854184255453, reg_lambda=1.5846072919687972, subsample=0.8693169523079376; total time=  18.1s
[CV] END colsample_bylevel=0.6481245808362168, colsample_bytree=0.785440629403996, gamma=0.35025252522341144, learning_rate=0.12841138846463387, max_depth=4, min_child_weight=9, n_estimators=375, reg_alpha=0.5248854184255453, reg_lambda=1.5846072919687972, subsample=0.8693169523079376; total time=  18.3s
[CV] END colsample_bylevel=0.6481245808362168, colsample_bytree=0.785440629403996, gamma=0.35025252522341144, learning_rate=0.12841138846463387, max_depth=4, min_child_weight=9, n_estimators=375, reg_alpha=0.5248854184255453, reg_lambda=1.5846072919687972, subsample=0.8693

/Users/jonaskarlsen/Documents/git/haml-et/.venv/lib/python3.13/site-packages/joblib/externals/loky/process_executor.py:787: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


[CV] END colsample_bylevel=0.667576028149256, colsample_bytree=0.7057845652718968, gamma=0.12752169662069324, learning_rate=0.04563342517229488, max_depth=3, min_child_weight=1, n_estimators=451, reg_alpha=2.2801651603842776, reg_lambda=4.319525358019028, subsample=0.6374860973265067; total time=  32.0s
[CV] END colsample_bylevel=0.667576028149256, colsample_bytree=0.7057845652718968, gamma=0.12752169662069324, learning_rate=0.04563342517229488, max_depth=3, min_child_weight=1, n_estimators=451, reg_alpha=2.2801651603842776, reg_lambda=4.319525358019028, subsample=0.6374860973265067; total time=  31.2s
[CV] END colsample_bylevel=0.7288583480241967, colsample_bytree=0.7834757452229892, gamma=0.19350921158975343, learning_rate=0.07975427370859701, max_depth=4, min_child_weight=8, n_estimators=203, reg_alpha=0.1599385611830817, reg_lambda=2.524555496740907, subsample=0.7028368073457518; total time=  16.3s
[CV] END colsample_bylevel=0.7288583480241967, colsample_bytree=0.7834757452229892, 